# 00 · Environment check (run once on JupyterHub)

1. Installs the missing libraries into **this kernel's** environment.
2. Reports RAM / CPU / GPU / disk.
3. Confirms the data files are in place and readable.
4. Checks whether the hub can download the sentence-embedding model (needed for Phase 2).

**After the install cell finishes, restart the kernel (Kernel → Restart) and run the rest.**

In [ ]:
import sys, subprocess, importlib.util

# import name -> pip name
PKGS = {'pyarrow': 'pyarrow', 'psutil': 'psutil', 'sklearn': 'scikit-learn', 'matplotlib': 'matplotlib',
        'lightgbm': 'lightgbm', 'xgboost': 'xgboost', 'shap': 'shap', 'lifelines': 'lifelines', 'econml': 'econml'}

def pip_install(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], capture_output=True, text=True)
    print(('OK   ' if r.returncode == 0 else 'FAIL ') + ' '.join(args))
    if r.returncode != 0:
        print(r.stderr[-1500:])
    return r.returncode == 0

for mod, pkg in PKGS.items():
    if importlib.util.find_spec(mod) is None:
        pip_install(pkg)      # installed one at a time so a single failure (e.g. econml) doesn't block the rest
    else:
        print('have', mod)

# torch: install the small CPU build only if torch is missing (the default Linux wheel is ~2.5 GB with CUDA)
if importlib.util.find_spec('torch') is None:
    pip_install('torch', '--index-url', 'https://download.pytorch.org/whl/cpu')
else:
    print('have torch')
if importlib.util.find_spec('sentence_transformers') is None:
    pip_install('sentence-transformers')
else:
    print('have sentence_transformers')

print('\n>>> If anything was installed above: Kernel -> Restart, then run the cells below.')

In [ ]:
# (kernel restarted: nothing from the install cell is in memory)
import sys, importlib, platform
MODULES = ['pyarrow', 'psutil', 'sklearn', 'matplotlib', 'lightgbm', 'xgboost', 'shap', 'lifelines', 'econml',
           'pandas', 'numpy', 'torch', 'sentence_transformers']
print('python', platform.python_version(), '|', sys.executable)
for mod in MODULES:
    try:
        m = importlib.import_module(mod)
        print(f'  {mod:<22}{getattr(m, "__version__", "ok")}')
    except Exception as e:
        print(f'  {mod:<22}MISSING ({type(e).__name__})')

In [ ]:
import os, sys, shutil, psutil
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
vm = psutil.virtual_memory()
print(f'CPU cores      : {os.cpu_count()}')
print(f'RAM total/avail: {vm.total/1e9:.1f} GB / {vm.available/1e9:.1f} GB')
# Containers often report the host's RAM; the cgroup limit is the real ceiling.
for p in ['/sys/fs/cgroup/memory.max', '/sys/fs/cgroup/memory/memory.limit_in_bytes']:
    try:
        v = open(p).read().strip()
        print(f'cgroup limit   : {v if v == "max" else f"{int(v)/1e9:.1f} GB"}  ({p})')
    except Exception:
        pass
du = shutil.disk_usage(ROOT)
print(f'disk free      : {du.free/1e9:.1f} GB (at {ROOT})')
try:
    import torch
    print('GPU            :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU only)')
except Exception:
    print('GPU            : unknown (torch missing)')

In [ ]:
sys.path.insert(0, str(ROOT))
import pandas as pd
from src import features as F

cfg = F.get_config()
for name in [F.JOBS_FILE, F.SALARY_FILE]:
    p = F.find_raw_file(name, cfg)
    head = pd.read_csv(p, nrows=5)
    print(f'{name}: {p}  ({p.stat().st_size/1e6:,.0f} MB)  columns={len(head.columns)}')
print('\nprocessed dir:', cfg['DATA_PROC'])
print('figures dir  :', cfg['FIG_DIR'])

In [ ]:
# Can we download the embedding model? (Phase 2 falls back to TF-IDF if not.)
try:
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    v = m.encode(['registered nurse', 'rn - med surg', 'line cook'])
    sims = (v @ v.T)
    print('embedding model OK, dim =', v.shape[1])
    print('similarity nurse~rn = %.2f, nurse~cook = %.2f' % (sims[0, 1], sims[0, 2]))
except Exception as e:
    print('embedding model NOT available -> Phase 2 will use the TF-IDF fallback.')
    print(type(e).__name__, str(e)[:300])

**Record the results** (RAM, GPU, whether the model downloaded) — they decide whether Phase 2 uses embeddings or TF-IDF,
and whether the Cox model / causal forest need subsampling. Next: `01_build_dataset.ipynb`.